# Day 8 — PyTorch Data Pipelines

**Agri-AI Bootcamp** · Phase 2 of 5: *Deep Learning & PyTorch* · Days 5–8

> Write a custom Dataset for the tabular farm data, load batches of 32 and print per-batch status.

**Runtime:** `Runtime -> Change runtime type -> T4 GPU` before you start.

**How to use this notebook:** run the cells top to bottom. The *Worked solution*
section is complete and runnable; the *Your turn* section at the end is where you
experiment. Datasets that need a Kaggle/Roboflow key are simulated here with the
same columns and shapes, and the real download command is shown in a comment —
swap it in when you have the credentials. Example numbers are illustrative.

## 1. Explained

A Dataset says how to fetch one sample (__len__ and __getitem__). A DataLoader groups samples into mini-batches, shuffles them and can load in parallel. Mini-batches make each step cheaper, and slightly noisy gradients often help.

### 🌾 Agriculture example

A Dataset reads sensor_readings.csv, normalises columns and returns (features, label); with batch_size=32, 1,000 rows give 32 batches per epoch.

> **Watch out:** Fit scalers on training data only, then reuse them for validation and test.

## 2. Setup

In [ ]:
import torch, torch.nn as nn, numpy as np, pandas as pd
from torch.utils.data import Dataset, DataLoader, random_split
dev = "cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(0)

## 3. Data

In [ ]:
# ---- Write a real CSV to disk, then read it the way a Dataset would ---------
rng = np.random.default_rng(3)
n = 1000
sensors = pd.DataFrame(dict(
    moisture=rng.uniform(5, 45, n), temp=rng.uniform(18, 42, n),
    humidity=rng.uniform(20, 95, n), N=rng.uniform(20, 140, n), pH=rng.uniform(5.2, 8.0, n)))
score = (-0.3*sensors.moisture + 0.1*sensors.temp - 0.04*sensors.humidity + 4.5)
sensors["needs_water"] = (rng.random(n) < 1/(1+np.exp(-score))).astype(int)
sensors.to_csv("sensor_readings.csv", index=False)
print("wrote sensor_readings.csv", sensors.shape)
sensors.head()

## 4. Starter code from the course

Reference only — the shortest form of today's idea. It assumes variables that the
next section defines properly, so read it, don't run it. Section 5 is the runnable
version.

```python
from torch.utils.data import Dataset, DataLoader
class FarmDS(Dataset):
    def __init__(s,X,y): s.X,s.y=X,y
    def __len__(s): return len(s.X)
    def __getitem__(s,i): return s.X[i],s.y[i]
dl=DataLoader(FarmDS(X,y),batch_size=32,shuffle=True)
for i,(xb,yb) in enumerate(dl): print("batch",i,xb.shape)
```

## 5. Worked solution

In [ ]:
FEATS = ["moisture", "temp", "humidity", "N", "pH"]

class FarmDS(Dataset):
    """Reads the CSV once, hands back one (features, label) pair at a time."""
    def __init__(self, csv, rows=None, mu=None, sd=None):
        df = pd.read_csv(csv)
        if rows is not None: df = df.iloc[rows]
        self.X = df[FEATS].to_numpy(np.float32)
        self.y = df["needs_water"].to_numpy(np.int64)
        self.mu = self.X.mean(0) if mu is None else mu      # scaler fitted on TRAIN
        self.sd = self.X.std(0)  if sd is None else sd
        self.X = (self.X - self.mu) / self.sd
    def __len__(self):  return len(self.y)
    def __getitem__(self, i): return torch.from_numpy(self.X[i]), self.y[i]

idx = np.random.default_rng(0).permutation(1000)
cut = 800
train_ds = FarmDS("sensor_readings.csv", idx[:cut])
test_ds  = FarmDS("sensor_readings.csv", idx[cut:], mu=train_ds.mu, sd=train_ds.sd)
print(f"train {len(train_ds)} | test {len(test_ds)} | one sample: "
      f"{train_ds[0][0].shape} -> label {train_ds[0][1]}")
print("scaler reused on test (mu from train):", train_ds.mu.round(2))

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, drop_last=False)
test_dl  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=2)
print(f"\n{len(train_dl)} batches per epoch of {len(train_ds)} rows at batch_size=32")
for i, (xb, yb) in enumerate(train_dl):
    print(f"  batch {i}: x {tuple(xb.shape)} y {tuple(yb.shape)} positives {yb.sum().item()}")
    if i == 2: break

# ---- Train with mini-batches -------------------------------------------------
net   = nn.Sequential(nn.Linear(5,32), nn.ReLU(), nn.Linear(32,2)).to(dev)
opt   = torch.optim.Adam(net.parameters(), 1e-3)
lossf = nn.CrossEntropyLoss()
for epoch in range(15):
    net.train(); tot = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(dev), yb.to(dev)
        opt.zero_grad(); l = lossf(net(xb), yb); l.backward(); opt.step()
        tot += l.item() * len(yb)
    net.eval(); correct = 0
    with torch.no_grad():
        for xb, yb in test_dl:
            correct += (net(xb.to(dev)).argmax(1).cpu() == yb).sum().item()
    if epoch % 3 == 0 or epoch == 14:
        print(f"epoch {epoch:>2} train loss {tot/len(train_ds):.4f} | test acc {correct/len(test_ds):.3f}")

# ---- Leakage demo: fitting the scaler on EVERYTHING --------------------------
leaky = FarmDS("sensor_readings.csv")            # mu/sd from all 1000 rows
print(f"\ntrain-only mu[0] {train_ds.mu[0]:.3f} vs all-rows mu[0] {leaky.mu[0]:.3f}")
print("Small here, but on real sensor data this leak inflates your score silently.")

## 6. Your turn

1. Try batch_size 8, 32 and 256 — note the batches per epoch and the wall time.
2. Add a `transform` argument to FarmDS that can add Gaussian sensor noise during training only.
3. Use `random_split` instead of index slicing and say why reusing the train scaler is still your job.

In [ ]:
# TODO 1: batch-size sweep
import time
for bs in [8, 32, 256]:
    dl = DataLoader(train_ds, batch_size=bs, shuffle=True)
    t0 = time.time(); n_b = 0
    for xb, yb in dl: n_b += 1
    print(f"bs {bs:>3}: {n_b} batches, one pass in {time.time()-t0:.3f}s")

# TODO 2: noise transform inside __getitem__ (apply only when self.train is True)
# TODO 3: random_split version

## 7. Dataset and references

**Dataset:** [Kaggle: Crop Recommendation (N, P, K, pH, rainfall)](https://www.kaggle.com/datasets/atharvaingle/crop-recommendation-dataset)

**References**

- [PyTorch Datasets & DataLoaders](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html)

**Course material:** PyTorch Datasets & DataLoaders guide

---
**Next:** Day 9 — Spatial Feature Extraction with CNNs